## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [28]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [29]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [30]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [31]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [32]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [ ]:
# Write collection_bbox(features) and apply to all four LOD files

def collection_bbox(features):
    lon_min = float("inf")
    lat_min = float("inf")
    lon_max = float("-inf")
    lat_max = float("-inf")

    for f in features:
        bbox = feature_bbox(f)
        lon_min = min(lon_min, bbox[0])
        lat_min = min(lat_min, bbox[1])
        lon_max = max(lon_max, bbox[2])
        lat_max = max(lat_max, bbox[3])

    return [lon_min, lat_min, lon_max, lat_max]


lod_files = {
    "coarse":     "../../data/lod/railroads_coarse.geojson",
    "medium":     "../../data/lod/railroads_medium.geojson",
    "fine":       "../../data/lod/railroads_fine.geojson",
    "extra_fine": "../../data/lod/railroads_extra_fine.geojson",
}

print(f"{'Level':<12} {'Collection bbox'}")
print("-" * 60)

for name, path_str in lod_files.items():
    lod_path = Path(path_str)
    with open(lod_path) as f:
        fc = json.load(f)

    bbox = collection_bbox(fc["features"])
    print(f"{name:<12} {bbox}")

# The medium, fine, and extra_fine levels should cover the same overall extent,
# because they all come from the full railroad dataset.
# The coarse level may be slightly smaller, because it uses the scalerank <= 4 filter.

Level        Collection bbox
------------------------------------------------------------
coarse       [-123.014722, -41.475186, 150.961667, 60.976516]
medium       [-150.112222, -51.894722, 179.357778, 69.604375]
fine         [-150.112222, -51.894722, 179.357778, 69.604375]
extra_fine   [-150.112222, -51.895278, 179.357778, 69.604375]


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [36]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson

lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

rows = []

for f in fine["features"]:
    lon_min, lat_min, lon_max, lat_max = feature_bbox(f)
    area = (lon_max - lon_min) * (lat_max - lat_min)
    category = f["properties"]["category"]
    rows.append((area, category))

rows.sort(reverse=True)

print(f"{'Rank':>4} {'BBox area':>12} {'Category':>10}")
print("-" * 30)

for i, (area, category) in enumerate(rows[:5], start=1):
    print(f"{i:>4} {area:>12.3f} {str(category):>10}")

# These results should make geographic sense.
# The largest bounding boxes should belong to long railroad features
# that span a large geographic area, not short local segments.

Rank    BBox area   Category
------------------------------
   1       29.312          0
   2       18.649          0
   3       17.704          3
   4       17.387          2
   5       12.355          2


## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?
a---

Two train tracks could have the same bounding box if one goes straight diagonally across and the other zigzags back and forth but ends at the same corners. They reach the same min/max longitude and latitude, so the box is identical.

It doesn't cause a problem for the culling system. The bounding box is just a quick first filter.

## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.